In [1]:
!pip install pandas nltk

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 525.4 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 8.0 MB/s eta 0:00:00:00:01


In [2]:
import pandas as pd
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to /home/ucloud/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /home/ucloud/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/ucloud/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/ucloud/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt to /home/ucloud/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/ucloud/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/ucloud/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [4]:
# load the cleaned data
clean_df = pd.read_csv('final_dataset_cleaned.csv')

In [ ]:
# shape of the dataset
print("Shape of the dataset:", clean_df.shape)

Shape of the dataset: (40301, 41)


In [ ]:
# Boilerplate phrases
boilerplate_phrases = [
    "Risks and challenges",
    "Learn about accountability on Kickstarter"
]

# Regex pattern that matches the exact phrases
pattern = r"|".join(re.escape(p) for p in boilerplate_phrases)

# Filter rows that contain any of these phrases exactly
boilerplate_rows = clean_df[
    clean_df["risks_html_text"].str.contains(pattern, na=False)
]

# Show the rows and their index
print(boilerplate_rows[["risks_html_text"]])
print(f"Total rows with leftover boilerplate: {len(boilerplate_rows)}")

                                         risks_html_text
228    Risks and challenges As I have done two rounds...
265    Risks and challenges The obvious risks and cha...
304    Risks and challenges We've been working on Vyu...
325    Risks and challenges Nowaday on the project wo...
341    Risks and challenges We have decided to use a ...
...                                                  ...
39777  Risks and challenges This is our third project...
39818  Risks and challenges We have various suppliers...
39843  Risks and challenges With our years of experie...
39847  Risks and challenges There are little or no ri...
39945  Risks and challenges The PHD-42 is a Personal ...

[1076 rows x 1 columns]

Total rows with leftover boilerplate: 1076


In [9]:
# remove leftover boilerplates from risks_html_text
def remove_boilerplate(text):
    if pd.isna(text):
        return text
    for phrase in boilerplate_phrases:
        text = re.sub(re.escape(phrase), "", text, flags=re.IGNORECASE)
    return text.strip()
clean_df["risks_html_text"] = clean_df["risks_html_text"].apply(remove_boilerplate)

In [ ]:
# NLP preprocessing
stop_words_list = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Function to map NLTK POS tags to WordNet POS tags for lemmatization
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# Function to clean text for NLP
def clean_for_nlp(text):
    if pd.isna(text):
        return ""
    # Lowercase the text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+|\b\S+\.[a-z]{2,}\S*\b', ' ', text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', ' ', text)
    # Remove punctuation (keep word characters and spaces)
    text = re.sub(r'[^\w\s]', ' ', text)
    # Remove standalone numbers
    text = re.sub(r'\b\d+\b', ' ', text)
    # Normalize whitespace (ONLY at end)
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize, remove stop words, and lemmatize
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words_list]
    tagged = nltk.pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    return " ".join(lemmatized)

In [12]:
# LLM preprocessing
def clean_for_llm(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Define leakage words and phrases
leakage_words = [
    'successfully funded','fully funded',
    'campaign failed','campaign succeeded',
    'not successful','not funded','not fully funded',
    'campaign funded','campaign is successful','campaign is fully funded',

    'reached our goal','goal reached','goal achieved', 'goal completed', 
    'reached our funding goal','funding goal reached',
    'reached our target','target reached','target achieved',
    'met our goal','goal met',

    'did not reach',"didn't reach",
    'did not meet',"didn't meet",
    'we made it',"we didn't make it",
    'we did it',"we didn't do it",

    'stretch goal unlocked','stretch goal reached','stretch goal achieved',

    'thanks for backing','thanks for pledging',
    'thank you for backing','thank you for pledging',

    '% funded','% of goal','% of funding goal',
    'percent funded','percent of goal','percent of funding goal',

    'over funded','under funded','half funded','partially funded',
    'almost funded','nearly funded','just funded','barely funded',

    'close to funding','close to goal','close to success',

    'final chance','last chance','final day','last day',
    'few days left','hours left','minutes left','seconds left',

    'campaign ended','campaign ending','campaign closed','campaign closing',
    'project funded', 'project fully funded', 'funding complete', 'funding achieved', 'funding successful',
    'passed our goal', 'exceeded our goal', 'campaign completed'
]

# ensure longer phrases match first
leakage_words = sorted(leakage_words, key=len, reverse=True)

# regex pattern that matches the exact phrases
LEAKAGE_PATTERN = re.compile(
    r'(' + '|'.join(re.escape(word) for word in leakage_words) + r')',
    flags=re.IGNORECASE
)

# Function to remove leakage words from text
def remove_leakage_words(text):
    if text is None or not isinstance(text, str):
        return text

    cleaned = LEAKAGE_PATTERN.sub('', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

In [14]:
# define textual features to process
textual_features = ['blurb', 'name', 'story_html_text', 'risks_html_text', 'rewards_html_text']

In [15]:
# apply to col in textual_features without adding new columns
for col in textual_features:
    clean_df[col + '_nlp'] = clean_df[col].apply(remove_leakage_words)

clean_df[[c + '_nlp' for c in textual_features]].head()

,blurb_nlp,name_nlp,story_html_text_nlp,risks_html_text_nlp,rewards_html_text_nlp
0,"Alumni-owned, community-powered craft coffee s...",Monarch Espresso Bar,We met in Tuscaloosa. We fell in love in Tusca...,The challenges that come with our project are ...,$10 Free cup of coffee at Monarch $25 Printed ...
1,"An easy-to-play, laugh-a-lot card game for you...",Ferret Flush®,THE GAME Ferret Flush® is a card game for crea...,CHALLENGES 1. Funding to print each edition of...,Starter Pack $25 The Ferret Flush® Original pa...
2,Oriented Air Conduction | Open-ear | Stereo Su...,Hakii Mix: Wearable Stereo Open Ear Headphone,Testimonials How to Operate the Hakii Mix Abou...,Timing is always a challenge with the producti...,Hakii Mix - Early Bird x 1 $109 Get your hands...
3,BodEball is a futuristic smartball that lights...,BodEBall – The Ball of the Future,BodEBall – The Ball of the Future BodEball is ...,NaN,donation $5 Donate and your name will appear o...
4,A grizzled ex-outlaw takes a risky escort miss...,Dragon Grit #1-4: A Fantasy Western With Dragons!,Who will love Dragon Grit? Fans of: gritty Wes...,I have and fulfilled over 30 comic book Kickst...,Digital #4 $8 24 pages of western fantasy adve...


In [ ]:
# remove leakage words for LLM preprocessing
for col in textual_features:
    clean_df[col + '_llm'] = clean_df[col].apply(remove_leakage_words)

clean_df[[c + '_llm' for c in textual_features]].head()

,blurb_llm,name_llm,story_html_text_llm,risks_html_text_llm,rewards_html_text_llm
0,"Alumni-owned, community-powered craft coffee s...",Monarch Espresso Bar,We met in Tuscaloosa. We fell in love in Tusca...,The challenges that come with our project are ...,$10 Free cup of coffee at Monarch $25 Printed ...
1,"An easy-to-play, laugh-a-lot card game for you...",Ferret Flush®,THE GAME Ferret Flush® is a card game for crea...,CHALLENGES 1. Funding to print each edition of...,Starter Pack $25 The Ferret Flush® Original pa...
2,Oriented Air Conduction | Open-ear | Stereo Su...,Hakii Mix: Wearable Stereo Open Ear Headphone,Testimonials How to Operate the Hakii Mix Abou...,Timing is always a challenge with the producti...,Hakii Mix - Early Bird x 1 $109 Get your hands...
3,BodEball is a futuristic smartball that lights...,BodEBall – The Ball of the Future,BodEBall – The Ball of the Future BodEball is ...,NaN,donation $5 Donate and your name will appear o...
4,A grizzled ex-outlaw takes a risky escort miss...,Dragon Grit #1-4: A Fantasy Western With Dragons!,Who will love Dragon Grit? Fans of: gritty Wes...,I have and fulfilled over 30 comic book Kickst...,Digital #4 $8 24 pages of western fantasy adve...


In [17]:
# apply NLP and LLM cleaning
for col in textual_features:
    clean_df[col + '_nlp'] = clean_df[col + '_nlp'].apply(clean_for_nlp)
    clean_df[col + '_llm'] = clean_df[col + '_llm'].apply(clean_for_llm)

In [18]:
clean_df[[c + '_nlp' for c in textual_features]].head()

,blurb_nlp,name_nlp,story_html_text_nlp,risks_html_text_nlp,rewards_html_text_nlp
0,alumnus own community power craft coffee shop ...,monarch espresso bar,meet tuscaloosa fell love tuscaloosa fell love...,challenge come project unforeseen update need ...,free cup coffee monarch print monarch mug choc...
1,easy play laugh lot card game emotional well m...,ferret flush,game ferret flush card game creature like dig ...,challenge fund print edition card game print u...,starter pack ferret flush original pack design...
2,orient air conduction open ear stereo surround...,hakii mix wearable stereo open ear headphone,testimonial operate hakii mix u hakii brand fo...,timing always challenge production new product...,hakii mix early bird x get hand hakii mix reta...
3,bodeball futuristic smartball light different ...,bodeball ball future,bodeball ball future bodeball futuristic smart...,,donation donate name appear special thanks pag...
4,grizzle ex outlaw take risky escort mission ac...,dragon grit fantasy western dragon,love dragon grit fan gritty western story drag...,fulfil comic book kickstarters good relationsh...,digital page western fantasy adventure digital...


In [19]:
# check results
clean_df[[c + '_llm' for c in textual_features]].head()

,blurb_llm,name_llm,story_html_text_llm,risks_html_text_llm,rewards_html_text_llm
0,"Alumni-owned, community-powered craft coffee s...",Monarch Espresso Bar,We met in Tuscaloosa. We fell in love in Tusca...,The challenges that come with our project are ...,$10 Free cup of coffee at Monarch $25 Printed ...
1,"An easy-to-play, laugh-a-lot card game for you...",Ferret Flush®,THE GAME Ferret Flush® is a card game for crea...,CHALLENGES 1. Funding to print each edition of...,Starter Pack $25 The Ferret Flush® Original pa...
2,Oriented Air Conduction | Open-ear | Stereo Su...,Hakii Mix: Wearable Stereo Open Ear Headphone,Testimonials How to Operate the Hakii Mix Abou...,Timing is always a challenge with the producti...,Hakii Mix - Early Bird x 1 $109 Get your hands...
3,BodEball is a futuristic smartball that lights...,BodEBall – The Ball of the Future,BodEBall – The Ball of the Future BodEball is ...,,donation $5 Donate and your name will appear o...
4,A grizzled ex-outlaw takes a risky escort miss...,Dragon Grit #1-4: A Fantasy Western With Dragons!,Who will love Dragon Grit? Fans of: gritty Wes...,I have and fulfilled over 30 comic book Kickst...,Digital #4 $8 24 pages of western fantasy adve...


In [20]:
# save 
clean_df.to_csv('final_dataset_preprocessed_final.csv', index=False)